# Setup environment

In [23]:
%pip install -q "crewai[tools, agentops]==0.114.0"
%pip install -q python-dotenv

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [24]:
from crewai import Agent, Task, Crew, Process, LLM
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import List, Optional
import agentops
import os

load_dotenv()
agentsops_api_key = os.getenv("AGENTSOPS_API_KEY")
llm_api_key = os.getenv("GEMINI_API_KEY")

agentops.init(
    api_key = agentsops_api_key,
    skip_auto_end_session = True,
    default_tags = ['crewai']
)

basic_llm = LLM(
    model = "gemini/gemini-3.6-flash",
    api_key = llm_api_key,
    temperature = 0
)

In [25]:
output_dir = "./ai-agent-output"
os.makedirs(output_dir, exist_ok=True)

# Setup Agents

In [26]:
no_keywords = 10

# example output from the agent
{
    "queries" : [
        "keyword1", "keyword2", "keyword3"
    ]
}

{'queries': ['keyword1', 'keyword2', 'keyword3']}

In [27]:
class SuggestedSearchQueries(BaseModel):
    queries: List[str] = Field(...,   # for required field
                               description = "A list of suggested search queries to be passed to the search engine.",
                               min_length = 1,
                               max_length = no_keywords
                        )

search_queries_recommendation_agent = Agent(
    role = "Search Queries Recommendation Agent",
    goal = "\n".join([
        "To provide a list of suggested search queries to be passed to the search engine.",
        "The queries must be varied and looking for specific items."
    ]),
    backstory = "The agent is designed to help in looking for products by providing a list of suggested search queries to be passed to the search engine based on the context provided.",
    llm = basic_llm,
    verbose = True
)

search_queries_recommendation_task = Task(
    description = "\n".join([
        "Rankyx is looking to buy {product_name} at the best prices (value for a price strategy)",
        "The company target any of these websites to buy from: {websites_list}",
        "The company wants to reach all available proucts on the internet to be compared later in another stage.",
        "The stores must sell the product in {country_name}",
        "Generate at maximum {no_keywords} queries.",
        "The search keywords must be in {language} language.",
        "Search keywords must contains specific brands, types or technologies. Avoid general keywords.",
        "The search query must reach an ecommerce webpage for product, and not a blog or listing page."
    ]),
    expected_output = "A JSON object containing a list of suggested search queries.",
    output_json = SuggestedSearchQueries,
    output_file = os.path.join(output_dir, "step_1_suggested_search_queries.json"),
    agent = search_queries_recommendation_agent
)

In [28]:
rankyx_crew = Crew(
    agents = [search_queries_recommendation_agent],
    tasks = [search_queries_recommendation_task],
    process = Process.sequential,
)

In [29]:
crew_results = rankyx_crew.kickoff(
    inputs={
        "product_name": "coffee machine for the office",
        "websites_list": ["www.amazon.eg", "www.jumia.com.eg", "www.noon.com/egypt-en"],
        "country_name": "Egypt",
        "no_keywords": 10,
        "language": "English"
    }
)

# Agent: Search Queries Recommendation Agent
## Task: Rankyx is looking to buy coffee machine for the office at the best prices (value for a price strategy)
The company target any of these websites to buy from: ['www.amazon.eg', 'www.jumia.com.eg', 'www.noon.com/egypt-en']
The company wants to reach all available proucts on the internet to be compared later in another stage.
The stores must sell the product in Egypt
Generate at maximum 10 queries.
The search keywords must be in English language.
Search keywords must contains specific brands, types or technologies. Avoid general keywords.
The search query must reach an ecommerce webpage for product, and not a blog or listing page.


# Agent: Search Queries Recommendation Agent
## Final Answer: 
{
  "queries": [
    "site:amazon.eg DeLonghi Dedica EC685 espresso coffee machine",
    "site:jumia.com.eg Black and Decker drip coffee maker 1.25L",
    "site:noon.com/egypt-en Tornado espresso cappuccino machine 15 bar",
    "site:amazon.eg Ph